# 04 – SQL Business Analysis

This notebook loads the cleaned datasets into an in-memory SQLite database and demonstrates business queries.

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/Cart2Insights")
CLEAN = ROOT / "data" / "cleaned"
RAW = ROOT / "data" / "raw"

if not CLEAN.exists():
    raise FileNotFoundError("Project data/cleaned folder was not found.")

print("Cleaned data folder:", CLEAN)


In [ ]:

import sqlite3

conn = sqlite3.connect(":memory:")

tables = {
    "customers": "customers_cleaned.csv",
    "orders": "orders_cleaned.csv",
    "order_items": "order_items_cleaned.csv",
    "payments": "payments_cleaned.csv",
    "reviews": "reviews_cleaned.csv",
    "products": "products_cleaned.csv",
    "sellers": "sellers_cleaned.csv"
}

for table, filename in tables.items():
    df = pd.read_csv(CLEAN / filename, low_memory=False)
    df.to_sql(table, conn, if_exists="replace", index=False)
    print(table, "loaded:", len(df), "rows")


In [ ]:

# 1. Revenue and order count
display(pd.read_sql_query(
    "SELECT COUNT(DISTINCT order_id) AS total_orders, "
    "ROUND(SUM(price), 2) AS item_revenue FROM order_items",
    conn
))


In [ ]:

# 2. Revenue by product category
query = '''
SELECT p.product_category_name,
       ROUND(SUM(i.price), 2) AS revenue
FROM order_items i
JOIN products p ON i.product_id = p.product_id
GROUP BY p.product_category_name
ORDER BY revenue DESC
LIMIT 10
'''
display(pd.read_sql_query(query, conn))


In [ ]:

# 3. CTE: customers with more than one order
query = '''
WITH customer_orders AS (
    SELECT customer_id, COUNT(*) AS order_count
    FROM orders
    GROUP BY customer_id
)
SELECT * FROM customer_orders
WHERE order_count > 1
ORDER BY order_count DESC
LIMIT 10
'''
display(pd.read_sql_query(query, conn))
